# Lesson 2.5 — Performance Optimization: EC2, EBS, RDS & S3

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Identify EC2 performance bottlenecks using CloudWatch metrics and Compute Optimizer
- Select the right EBS volume type for workload patterns
- Optimize RDS using Performance Insights and read replicas
- Implement S3 performance strategies (multipart upload, Transfer Acceleration, intelligent tiering)
- Right-size compute resources based on utilization data

**Exam relevance:** SOA-C03 Domain 1, Skills 1.3.1–1.3.6

_Reference: [EC2 Instance Types](https://aws.amazon.com/ec2/instance-types/) | [EBS Volume Types](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ebs-volume-types.html)_

## 1. EC2 Performance Optimization

### Instance Families (know these for exams!)

| Family | Optimized For | Use Cases |
|--------|--------------|----------|
| **t3/t4g** | Burstable | Dev/test, low-traffic web apps |
| **m5/m6i** | General purpose | Application servers, mid-size DBs |
| **c5/c6i** | Compute | CPU-intensive: encoding, analytics, gaming |
| **r5/r6i** | Memory | In-memory DBs, Spark, SAP HANA |
| **x1/x2** | High memory | SAP HANA, large in-memory workloads |
| **i3/i4i** | Storage I/O | NoSQL, data warehousing, log processing |
| **p3/p4d** | GPU | ML training, graphics rendering |
| **inf1/inf2** | ML Inference | Low-latency ML inference |
| **g4dn/g5** | GPU + local NVMe | ML inference + gaming |

**Graviton (ARM) = g suffix**: `m7g`, `c7g`, `r7g` — up to 40% better price/performance

### T-Series CPU Credits (Burstable)
- T3 instances earn **CPU credits** when below baseline CPU
- Spend credits when above baseline → up to 100% CPU burst
- When credits depleted → throttled to baseline
- **T3 Unlimited mode**: burst indefinitely, pay for surplus credits

### EC2 Placement Groups
| Type | Network | Use Case |
|------|---------|----------|
| **Cluster** | Lowest latency, high throughput (10 Gbps+) | HPC, big data, tightly coupled apps |
| **Spread** | Instances on separate hardware | Critical HA apps, max 7 instances/AZ |
| **Partition** | Groups of instances on separate racks | Hadoop, Cassandra, Kafka |

## 2. EBS Volume Types

| Type | IOPS | Throughput | Use Case |
|------|------|-----------|----------|
| **gp3** | Up to 16,000 | Up to 1,000 MB/s | General purpose (default choice) |
| **gp2** | 3 IOPS/GB, up to 16,000 | 250 MB/s | Legacy general purpose |
| **io2 Block Express** | Up to 256,000 | Up to 4,000 MB/s | Mission-critical, high I/O DBs |
| **io1** | Up to 64,000 | Up to 1,000 MB/s | I/O-intensive DBs (Oracle, SQL Server) |
| **st1** | Sequential burst | 500 MB/s | Big data, log processing, data warehouses |
| **sc1** | Sequential burst | 250 MB/s | Cold data, infrequently accessed |

**gp3 vs gp2**: Always prefer **gp3** — it's cheaper AND you can independently set IOPS and throughput

## 3. S3 Performance

- **Baseline**: 3,500 PUT/COPY/POST/DELETE and 5,500 GET/HEAD requests per second **per prefix**
- **Multipart Upload**: Required for objects > 5GB, recommended for > 100MB
- **S3 Transfer Acceleration**: Routes uploads through CloudFront edge locations (for cross-region)
- **S3 Intelligent-Tiering**: Automatically moves objects between access tiers based on usage patterns

In [ ]:
import boto3
import json

ec2 = boto3.client('ec2', region_name='us-east-1')
s3 = boto3.client('s3', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']

print("Setup complete.")

In [ ]:
# AWS Compute Optimizer — right-sizing recommendations
# Compute Optimizer analyzes CloudWatch metrics and recommends optimal instance types

optimizer = boto3.client('compute-optimizer', region_name='us-east-1')

try:
    status = optimizer.get_enrollment_status()
    print(f"Compute Optimizer enrollment: {status['status']}")
    
    if status['status'] == 'Active':
        # Get EC2 instance recommendations
        recs = optimizer.get_ec2_instance_recommendations()
        print(f"\nRecommendations ({len(recs['instanceRecommendations'])}):")
        for rec in recs['instanceRecommendations'][:3]:
            current = rec['currentInstanceType']
            finding = rec['finding']  # UNDER_PROVISIONED, OVER_PROVISIONED, OPTIMIZED
            print(f"  {rec['instanceArn'].split('/')[-1]} ({current}) — {finding}")
            for option in rec['recommendationOptions'][:2]:
                savings = option.get('estimatedMonthlySavings', {})
                print(f"    → Recommended: {option['instanceType']} (savings: ${savings.get('value', 0):.2f}/mo)")
    else:
        print("\nCompute Optimizer not enrolled. To enable:")
        print("  optimizer.update_enrollment_status(status='Active')")
        print("  Note: Needs 14+ days of CloudWatch data for accurate recommendations")
except Exception as e:
    print(f"Compute Optimizer: {e}")

In [ ]:
# Visualize EBS volume type selection decision tree
import json

ebs_decision = [
    {
        "scenario": "New application — general purpose workload",
        "recommendation": "gp3",
        "reason": "Best price/performance, independent IOPS/throughput tuning"
    },
    {
        "scenario": "Oracle DB requiring 50,000 IOPS consistently",
        "recommendation": "io2 Block Express",
        "reason": "Highest IOPS (256k), 99.999% durability, sub-millisecond latency"
    },
    {
        "scenario": "Data warehouse sequential reads of 100TB",
        "recommendation": "st1",
        "reason": "Optimized for sequential I/O, 500 MB/s throughput, low cost"
    },
    {
        "scenario": "Cold archive data accessed monthly",
        "recommendation": "sc1 or S3 Glacier",
        "reason": "Lowest cost for infrequent access; consider S3 if not block storage needed"
    },
    {
        "scenario": "Boot volume for EC2 instance",
        "recommendation": "gp3",
        "reason": "Balanced performance for OS, fast boot time, cost-effective"
    }
]

print("EBS Volume Type Selection Guide:\n")
for i, d in enumerate(ebs_decision, 1):
    print(f"{i}. Scenario: {d['scenario']}")
    print(f"   Recommendation: {d['recommendation']}")
    print(f"   Reason: {d['reason']}")
    print()

In [ ]:
# S3 Performance: Demonstrate multipart upload strategy
import boto3
import io

s3 = boto3.client('s3', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
BUCKET_NAME = f'cloudops-perf-demo-{account_id}'

# Create a demo bucket
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_encryption(
        Bucket=BUCKET_NAME,
        ServerSideEncryptionConfiguration={
            'Rules': [{'ApplyServerSideEncryptionByDefault': {'SSEAlgorithm': 'AES256'}}]
        }
    )
    print(f"Created bucket: {BUCKET_NAME}")
except Exception as e:
    print(f"Bucket: {e}")

# Enable Intelligent-Tiering for automatic cost optimization
s3.put_bucket_intelligent_tiering_configuration(
    Bucket=BUCKET_NAME,
    Id='EntireBucket',
    IntelligentTieringConfiguration={
        'Id': 'EntireBucket',
        'Status': 'Enabled',
        'Tierings': [
            {'Days': 90, 'AccessTier': 'ARCHIVE_ACCESS'},     # Move to Archive after 90d without access
            {'Days': 180, 'AccessTier': 'DEEP_ARCHIVE_ACCESS'} # Deep Archive after 180d
        ]
    }
)
print("Enabled S3 Intelligent-Tiering")

# Demonstrate multipart upload (for large files)
# In production: use TransferManager which handles this automatically
from boto3.s3.transfer import TransferConfig

transfer_config = TransferConfig(
    multipart_threshold=1024 * 25,   # 25MB threshold
    max_concurrency=10,              # 10 parallel upload threads
    multipart_chunksize=1024 * 25,   # 25MB chunks
    use_threads=True
)

# Upload a small demo file (would use multipart for files > 25MB)
demo_data = b'x' * (1024 * 1024)  # 1MB demo object
s3.upload_fileobj(
    io.BytesIO(demo_data),
    BUCKET_NAME,
    'demo/large-object.bin',
    Config=transfer_config
)
print(f"Uploaded demo object to s3://{BUCKET_NAME}/demo/large-object.bin")
print("\nFor files > 100MB, TransferManager automatically uses multipart upload.")

In [ ]:
# S3 Lifecycle Policy — automatically transition objects to cheaper tiers

lifecycle_policy = {
    'Rules': [
        {
            'ID': 'TransitionLogsToGlacier',
            'Status': 'Enabled',
            'Filter': {'Prefix': 'logs/'},
            'Transitions': [
                {'Days': 30, 'StorageClass': 'STANDARD_IA'},      # After 30d → Infrequent Access
                {'Days': 90, 'StorageClass': 'GLACIER'},           # After 90d → Glacier
                {'Days': 365, 'StorageClass': 'DEEP_ARCHIVE'}      # After 1yr → Deep Archive
            ],
            'Expiration': {'Days': 2555}                           # Delete after 7 years
        },
        {
            'ID': 'CleanupIncompleteMultipartUploads',
            'Status': 'Enabled',
            'Filter': {'Prefix': ''},
            'AbortIncompleteMultipartUpload': {'DaysAfterInitiation': 7}  # IMPORTANT!
        }
    ]
}

s3.put_bucket_lifecycle_configuration(
    Bucket=BUCKET_NAME,
    LifecycleConfiguration=lifecycle_policy
)
print("Lifecycle policy applied:")
print("  logs/ → Standard-IA after 30d → Glacier after 90d → Deep Archive after 1yr → Delete after 7yr")
print("  Incomplete multipart uploads → Aborted after 7 days")

## Exercise

In [ ]:
# TODO: Write a function that audits S3 buckets and flags performance/cost issues:
# - Buckets without lifecycle policies
# - Buckets without encryption
# - Buckets with objects in STANDARD storage that haven't been accessed in 30+ days
#   (Use CloudWatch S3 metrics or S3 Storage Lens for last access)
# Print a report with recommendations for each flagged bucket

import boto3

def audit_s3_buckets() -> list:
    """Audit S3 buckets for performance and cost optimization opportunities."""
    s3 = boto3.client('s3', region_name='us-east-1')
    findings = []
    
    buckets = s3.list_buckets()['Buckets']
    
    for bucket in buckets:
        name = bucket['Name']
        bucket_findings = {'bucket': name, 'issues': []}
        
        # TODO: Check for encryption
        # Hint: s3.get_bucket_encryption(Bucket=name) raises ClientError if not enabled
        
        # TODO: Check for lifecycle policy
        # Hint: s3.get_bucket_lifecycle_configuration(Bucket=name) raises NoSuchLifecycleConfiguration
        
        if bucket_findings['issues']:
            findings.append(bucket_findings)
    
    return findings

results = audit_s3_buckets()
print(f"Found {len(results)} buckets with optimization opportunities:")
for r in results:
    print(f"\n  Bucket: {r['bucket']}")
    for issue in r['issues']:
        print(f"    - {issue}")

In [ ]:
# Cleanup
import boto3

s3 = boto3.client('s3', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
BUCKET_NAME = f'cloudops-perf-demo-{account_id}'

try:
    # Delete all objects first
    objects = s3.list_objects_v2(Bucket=BUCKET_NAME).get('Contents', [])
    if objects:
        s3.delete_objects(Bucket=BUCKET_NAME, Delete={'Objects': [{'Key': o['Key']} for o in objects]})
    s3.delete_bucket(Bucket=BUCKET_NAME)
    print(f"Deleted bucket: {BUCKET_NAME}")
except Exception as e:
    print(f"Bucket cleanup: {e}")

## Summary

- **EC2 right-sizing**: Use Compute Optimizer (needs 14+ days of CloudWatch data)
- **EBS**: Prefer `gp3` over `gp2`; use `io2` for high-IOPS databases; `st1` for sequential big data
- **Placement Groups**: Cluster (low latency), Spread (HA isolation), Partition (large distributed systems)
- **S3**: Multipart for large objects; Intelligent-Tiering for unpredictable access patterns; Lifecycle policies for cost management
- **Always abort incomplete multipart uploads** with a lifecycle rule (common cost leak!)

## Key Exam Points (SOA-C03)
- EC2 memory metrics require **CloudWatch Agent** — not available by default
- S3 requests scale **per prefix** — add prefix diversity to increase throughput beyond 5,500 GET/s
- RDS Performance Insights identifies **wait events** causing slow queries

---

**Next Phase:** [Phase 3 — Reliability & Business Continuity](../phase-3-reliability/lesson-3.1-auto-scaling-elb.ipynb)